Analysis of the agreement between the probabilities generated by the models and the actual frequencies (Calibration Curve) and the predictive stability of the Rank Averaging Ensemble.

In [4]:
import sys
import os
import importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from scripts import config
import scripts.utils.validation as validation
import scripts.utils.metrics as metrics
import scripts.utils.plotting as plotting
import scripts.model_trainer as trainer

importlib.reload(config)
importlib.reload(validation)
importlib.reload(metrics)
importlib.reload(plotting)
importlib.reload(trainer)

from scripts.config import STAGE4_OUT, STAGE5_OUT, RAW_DATA_DIR

plt.rcParams['figure.figsize'] = [12, 6]
sns.set_style("whitegrid")

print(f"Project root added: {project_root}")
print("Modules reloaded successfully.")

Project root added: c:\Users\Eren\Desktop\data_league_26
Modules reloaded successfully.


# **STAGE 5**

In [5]:
# Stage 4 çıktısını yükle
df_train = pd.read_parquet(os.path.join(STAGE4_OUT, "train_27.parquet"))

# Hedef ve meta kolonları belirle
target_col = 'label_noshow'
meta_cols = ['appointment_id', 'patient_id', 'clinic_id', 'label_noshow', 
             'appointment_datetime', 'booking_datetime', 'appointment_month']

# Modelin göreceği özellikleri (features) ayır
features = [c for c in df_train.columns if c not in meta_cols]

print(f"Dataset Size: {df_train.shape}")
print(f"Feature Count: {len(features)}")

Dataset Size: (179080, 33)
Feature Count: 27


In [7]:
# Eğer kolon yoksa, datetime üzerinden tekrar oluşturuyoruz
if 'appointment_month' not in df_train.columns:
    print("appointment_month found missing. Extracting from appointment_datetime...")
    # Datetime tipine dönüştürdüğümüzden emin olalım
    df_train['appointment_datetime'] = pd.to_datetime(df_train['appointment_datetime'])
    df_train['appointment_month'] = df_train['appointment_datetime'].dt.month

# Kontrol
print(f"Available columns: {df_train.columns.tolist()}")
print(f"Unique months in data: {df_train['appointment_month'].unique()}")

appointment_month found missing. Extracting from appointment_datetime...
Available columns: ['appointment_id', 'patient_id', 'clinic_id', 'label_noshow', 'appointment_datetime', 'booking_datetime', 'lead_time_hours', 'combo_te_area_id_appointment_type', 'lead_time_per_age', 'te_booking_channel_appointment_type', 'lead_time_per_distance', 'combo_te_specialty_area_id', 'sms_sent', 'combo_te_area_id_hour', 'combo_te_area_id_booking_channel', 'te_area_id_day_of_week', 'sms_lead_hours', 'distance_bucket', 'wait_mins_est', 'combo_te_specialty_booking_channel', 'combo_te_area_id_sex', 'clinic_load_ratio', 'combo_te_booking_channel_hour', 'te_specialty_hour', 'stat_clinic_id_wait_mins_est_mean', 'clinic_day_appt_count', 'combo_te_booking_channel_day_of_week', 'te_specialty_day_of_week', 'has_phone', 'stat_booking_channel_wait_mins_est_median', 'sms_policy_prob', 'clinic_distance_km_mean', 'clinic_lat', 'appointment_month']
Unique months in data: [1 4 5 6 7 8 2 9 3]


In [12]:
ve = validation.ValidationEngine()
splits = ve.get_expanding_window_splits(df_train, date_col='appointment_month', start_month=7, end_month=9)

# Kontrol
for i, (t_idx, v_idx) in enumerate(splits):
    val_month = df_train.iloc[v_idx]['appointment_month'].unique()[0]
    print(f"Fold {i+1}: Validating on Month {val_month} | Train size: {len(t_idx)} | Val size: {len(v_idx)}")

Fold 1: Validating on Month 7 | Train size: 118428 | Val size: 21228
Fold 2: Validating on Month 8 | Train size: 139656 | Val size: 19308
Fold 3: Validating on Month 9 | Train size: 158964 | Val size: 20116


In [13]:
custom_lgbm_params = {
    'objective': 'binary',
    'metric': 'average_precision', # Yarışma metriği
    'boosting_type': 'gbdt',
    'device': 'gpu',               # GPU kullanımı hızı artırır
    'random_state': 42,
    'learning_rate': 0.015,        # Daha yavaş ve stabil öğrenme
    'num_leaves': 42,              # Karmaşıklığı sınırladık
    'max_depth': 7,                # Overfitting'i engellemek için derinlik sınırı
    'is_unbalance': True,          # Sınıf dengesizliği yönetimi
    'lambda_l1': 1.5,              # L1 Regularization (Seyreklik sağlar)
    'lambda_l2': 2.0,              # L2 Regularization (Katsayıları dizginler)
    'feature_fraction': 0.7,       # Her ağaçta özelliklerin %70'ini rastgele seç
    'bagging_fraction': 0.7,       # Verinin %70'ini örnekle
    'bagging_freq': 5,             # Her 5 iterasyonda bir örnekleme yap
    'min_child_samples': 100,      # Bir yapraktaki minimum veri sayısı
    'verbose': -1
}

In [ ]:
import optuna
import json
from scripts.config import STAGE5_OUT

def objective(trial):
    # Defining the search space with rational boundaries
    params = {
        'objective': 'binary',
        'metric': 'average_precision',
        'boosting_type': 'gbdt',
        'device': 'gpu',
        'random_state': 42,
        'verbosity': -1,
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 5.0),        
        # Core hyperparameters to tune
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 500),
        
        # Regularization (Crucial for test set generalization)
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),

        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 50, 200),
        'pos_bagging_fraction': trial.suggest_float('pos_bagging_fraction', 0.5, 1.0),
        'neg_bagging_fraction': trial.suggest_float('neg_bagging_fraction', 0.5, 1.0),
        
        # Subsampling for variance reduction
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 0.9),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 0.9),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
    }

    fold_scores = []
    
    # Executing Expanding Window CV (3 Folds: July, August, September)
    for i in range(3):
        train_idx, val_idx = splits[i]
        
        X_train, y_train = df_train.iloc[train_idx][features], df_train.iloc[train_idx][target_col]
        X_val, y_val = df_train.iloc[val_idx][features], df_train.iloc[val_idx][target_col]
        
        # Initialize and Train
        lgbm_trainer = trainer.ModelTrainer(model_type='lgbm', params=params)
        lgbm_trainer.train(X_train, y_train, X_val, y_val)
        
        # Evaluate
        y_prob = lgbm_trainer.predict_proba(X_val)
        score = metrics.calculate_all_metrics(y_val, y_prob)['pr_auc']
        fold_scores.append(score)
    
    # Optimization Goal: Maximize Mean PR-AUC across folds
    return np.mean(fold_scores)

# Creating the study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50) # You can increase n_trials for more depth

print(f"\nOptimization Finished!")
print(f"Best PR-AUC Score: {study.best_value:.4f}")
print(f"Best Hyperparameters: {study.best_params}")

[I 2026-03-18 09:21:24,793] A new study created in memory with name: no-name-99b1970c-b9af-4835-bb2c-85be383118f1
[I 2026-03-18 09:21:36,292] Trial 0 finished with value: 0.4984430104303303 and parameters: {'scale_pos_weight': 1.9657536604971564, 'learning_rate': 0.04197085263411293, 'num_leaves': 34, 'max_depth': 14, 'min_child_samples': 494, 'lambda_l1': 2.980464898284213e-06, 'lambda_l2': 2.1357573270020967e-07, 'min_data_in_leaf': 167, 'pos_bagging_fraction': 0.9753027451875278, 'neg_bagging_fraction': 0.5557307387492532, 'feature_fraction': 0.417033905894221, 'bagging_fraction': 0.7358811593059684, 'bagging_freq': 5}. Best is trial 0 with value: 0.4984430104303303.
[I 2026-03-18 09:21:51,934] Trial 1 finished with value: 0.4880475383071993 and parameters: {'scale_pos_weight': 3.721214135550458, 'learning_rate': 0.0065547976414561055, 'num_leaves': 55, 'max_depth': 7, 'min_child_samples': 35, 'lambda_l1': 3.401634340651286e-06, 'lambda_l2': 6.832946604889349e-05, 'min_data_in_leaf'

# **STAGE 6**